# Week 10 experiments: which PPO change helps the drone fly through the hoop?

This notebook compares the runs started by `hardware/onboard/week10_experiments.py`. Each run starts from the **same parent policy** with the **same demos, seed, task and step budget**, and changes **one** thing:

| Run | What changes | Week 10 idea it tests |
|---|---|---|
| **A baseline** | nothing (SB3 defaults, like every run so far) | the control group |
| **B gamma** | γ 0.99 → 0.995, λ 0.95 → 0.97 | *Objective / horizon* (notes §4.1): γ=0.99 at 15 Hz only "sees" ~6.7 s ahead, but one lap takes 10–15 s |
| **C kl** | `target_kl=0.02`, batch 512, 5 epochs | *Trust regions* (notes §8.1, §9.6): stop each update before the policy moves too far |
| **D entropy** | `ent_coef=0.005`, log_std reset to −1 | *Entropy bonus* (notes §9.4): keep exploring instead of collapsing to std ≈ 0.08 |
| **E sde** | gSDE exploration | *Gaussian policies & exploration* (notes §10.1–10.2, SAC notebook `use_sde=True`) |
| **F lambda1** | GAE λ = 1.0, batch 512, 5 epochs | *Baselines add no bias* (notes §5.7): with Monte Carlo advantages a wrong critic can only add noise, never point the gradient the wrong way |

**What changed in the simulator first (2026-10-13), from 28 real Tello flights** (`hardware/privileged/reports/sim_fidelity.md`):
yaw is now clockwise-positive like the drone; the policy gets **no speed input** (the real Tello's speed readings are direction-less jitter in the cage, and the old policies leaned on them); `pushed` counts the forward commands actually sent; and the runs train on the **"measured"** profile (turning 0.40×, climbing 0.52×, a yaw dead zone). The parent is `onboard_20261011-032612`, the only older run that still passes (60 %) in this corrected sim.

Everything here is read straight from the run folders (`results/hoop/runs/w10_*`). The last section shows where to find **the same charts in ClearML**, so you can check one against the other.

## 1. Setup
Same idea as the class notebooks: imports, then where the logs live. Change `PATTERN` to `w10_laps_*` for stage 2.

In [ ]:
import os, sys, json, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

CMS = Path.cwd().resolve().parent if Path.cwd().name == "tutorial" else Path.cwd().resolve()
sys.path.insert(0, str(CMS / "hardware" / "onboard"))
RUNS_DIR = Path(os.environ.get("W10_RUNS_DIR", CMS / "results" / "hoop" / "runs"))
PATTERN = os.environ.get("W10_PATTERN", "w10_front_*")
run_dirs = sorted(d for d in RUNS_DIR.glob(PATTERN) if (d / "meta.json").exists())
print(f"{len(run_dirs)} runs in {RUNS_DIR} matching {PATTERN}:")
for d in run_dirs:
    print("  ", d.name)

## 2. Load every run
Each run folder has: `meta.json` (the settings), `passes.json` (an evaluation every 20,000 steps), `progress.csv` (everything SB3 prints during training: `train/std`, `train/approx_kl`, ...).

In [ ]:
def load_progress(path):
    cols = {}
    if not path.exists():
        return cols
    for row in csv.DictReader(open(path)):
        for k, v in row.items():
            try:
                cols.setdefault(k, []).append(float(v))
            except (TypeError, ValueError):
                cols.setdefault(k, []).append(np.nan)
    return {k: np.array(v) for k, v in cols.items()}

def load_run(d):
    meta = json.loads((d / "meta.json").read_text())
    passes = json.loads((d / "passes.json").read_text()) if (d / "passes.json").exists() else []
    prog = load_progress(d / "progress.csv")
    t0 = meta.get("init_timesteps") or 0
    return dict(name=d.name, dir=d, meta=meta, prog=prog,
                eval_t=np.array([p["t"] for p in passes]) - t0,
                eval_passes=np.array([p["mean"] for p in passes]),
                train_t=prog.get("time/total_timesteps", np.array([])) - t0)

runs = [load_run(d) for d in run_dirs]
for r in runs:
    print(f"{r['name']:<28} evals {len(r['eval_t']):>4}   training rows {len(r['train_t']):>5}   ppo {r['meta'].get('ppo')}")

## 3. The score: passes per evaluation episode
The same smoothing as `plot_results` in the PPO BipedalWalker notebook (a moving average), so one lucky evaluation does not decide the winner. **Higher is better.**

In [ ]:
def smooth(y, w=5):
    if len(y) < w:
        return y
    return np.convolve(y, np.ones(w) / w, mode="valid")

plt.figure(figsize=(11, 5))
for r in runs:
    y = smooth(r["eval_passes"])
    x = r["eval_t"][len(r["eval_t"]) - len(y):]
    plt.plot(x, y, label=r["name"])
plt.xlabel("steps since the run started")
plt.ylabel("mean passes per episode (smoothed)")
plt.title("Evaluation passes (no exploration noise)")
plt.legend()
plt.grid(alpha=.3)
plt.show()

## 4. Why it did (or didn't) work: the training diagnostics

| Chart | What it means (Week 10) | Healthy |
|---|---|---|
| `train/std` | Size of the Gaussian exploration noise (§10.1). Collapsing towards 0 means the policy has stopped exploring. | Falls slowly, doesn't flatline near 0.05 |
| `train/approx_kl` | How far each update moved the policy from the one that collected the data (§8.3). | ~0.005–0.02 |
| `train/clip_fraction` | Share of samples where PPO's clip kicked in (§9.2–9.3). High = the update wanted to move further than the trust region allows. | below ~0.1–0.15 |
| `train/explained_variance` | How well the critic predicts returns. Near 1 = good advantages, near 0 = the critic is guessing (§7). | rising towards 1 |

In [ ]:
keys = ["train/std", "train/approx_kl", "train/clip_fraction", "train/explained_variance"]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, k in zip(axes.flat, keys):
    for r in runs:
        y = r["prog"].get(k)
        if y is None or not len(y):
            continue
        ok = ~np.isnan(y)
        ax.plot(r["train_t"][ok], y[ok], label=r["name"], lw=1)
    ax.set_title(k)
    ax.set_xlabel("steps since the run started")
    ax.grid(alpha=.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Summary table
`best` = best smoothed evaluation, `last` = mean of the last 5 evaluations (what the run ended on), then the averages of the diagnostics over the second half of training.

In [ ]:
def half_mean(r, k):
    y = r["prog"].get(k)
    if y is None or not len(y):
        return np.nan
    y = y[~np.isnan(y)]
    return float(np.mean(y[len(y) // 2:])) if len(y) else np.nan

rows = []
for r in runs:
    sm = smooth(r["eval_passes"])
    rows.append(dict(run=r["name"],
                     steps=int(r["eval_t"][-1]) if len(r["eval_t"]) else 0,
                     best=float(sm.max()) if len(sm) else np.nan,
                     last=float(np.mean(r["eval_passes"][-5:])) if len(r["eval_passes"]) else np.nan,
                     std=half_mean(r, "train/std"), kl=half_mean(r, "train/approx_kl"),
                     clip=half_mean(r, "train/clip_fraction"), ev=half_mean(r, "train/explained_variance")))
rows.sort(key=lambda x: -np.nan_to_num(x["last"]))
head = "".join(f"<th>{h}</th>" for h in rows[0]) if rows else ""
body = "".join("<tr>" + "".join(f"<td>{v:.3f}</td>" if isinstance(v, float) else f"<td>{v}</td>"
                                for v in row.values()) + "</tr>" for row in rows)
display(HTML(f"<table><tr>{head}</tr>{body}</table>"))

## 6. The aliasing test (notes §3.1)

The ring looks the same from the front and the back, so some situations are **aliased**: two different places give the same 37 numbers. In the lecture's gridworld, a *deterministic* policy oscillates forever in aliased states, while a *stochastic* one gets through.

On the real Tello we fly the **mean** action (deterministic). Here every run's `best_model.zip` is scored both ways on the same layouts. If **noisy beats deterministic**, the observation is hiding something the policy needs (fixes: more memory in the observation, frame stacking as in the Atari notebook, or a recurrent policy). This takes a minute or two per run.

In [ ]:
from week10_experiments import evaluate

EPISODES = int(os.environ.get("W10_EVAL_EPISODES", 30))
STAGE = "laps" if "laps" in PATTERN else "front"
alias = []
for r in runs:
    z = r["dir"] / "best_model.zip"
    if not z.exists():
        continue
    det = evaluate(z, deterministic=True, episodes=EPISODES, stage=STAGE)
    sto = evaluate(z, deterministic=False, episodes=EPISODES, stage=STAGE)
    alias.append((r["name"], det, sto))
    print(f"{r['name']:<28} deterministic {det:.2f}   with noise {sto:.2f}   {'<- noise helps' if sto > det else ''}")

## 6b. How wrong is the critic? (notes §6.1, §7.8)

Run F exists because a learned critic can be wrong. This checks how wrong: each run's policy flies with its exploration noise, and at every state the critic's guess **V(s)** is compared with the discounted return **G** that actually followed. One return is an unbiased but noisy sample of V(s), so points scattered around the diagonal are noise; a cloud that is **tilted or shifted** off it is critic error. *Explained variance* near 1 = a good critic; *bias* > 0 = it is too optimistic.

In [ ]:
from week10_experiments import critic_check

fig, axes = plt.subplots(1, max(1, len(runs)), figsize=(4 * max(1, len(runs)), 4), squeeze=False)
for ax, r in zip(axes[0], runs):
    z = r["dir"] / "best_model.zip"
    if not z.exists():
        continue
    c = critic_check(z, episodes=int(os.environ.get("W10_CRITIC_EPISODES", 10)), stage=STAGE)
    ax.scatter(c["g"], c["v"], s=3, alpha=.3)
    lo, hi = float(min(c["g"].min(), c["v"].min())), float(max(c["g"].max(), c["v"].max()))
    ax.plot([lo, hi], [lo, hi], "k--", lw=1)
    ax.set_title(f"{r['name']}  |  EV {c['explained_variance']:.2f}, bias {c['bias']:+.1f}", fontsize=9)
    ax.set_xlabel("actual return G")
axes[0, 0].set_ylabel("critic V(s)")
plt.tight_layout()
plt.show()

## 7. Reading the result

* **One run clearly above A on the passes chart, and still there at the end** → that idea is worth keeping. Re-run it with a second seed (`--seed 1`) before trusting it: one seed can be luck.
* **B wins** → the critic was too short-sighted. Keep γ 0.995 for the laps stage.
* **C wins and A shows high `clip_fraction` / `approx_kl`** → updates were too big. This matches "more training made it worse".
* **D or E win and A's `std` collapses** → exploration was the bottleneck.
* **F wins** → the critic's bias was steering PPO wrong; section 6b should show A's critic tilted or shifted off the diagonal.
* **Nothing beats A** → the PPO settings aren't the problem. Look at the reward, the curriculum, or the sim-to-real gap (anatomy §9).

Then run stage 2 (`--stage laps --parent results/hoop/runs/<winner>/best_model.zip`) and point `PATTERN` at `w10_laps_*`.

## 8. The same comparison in ClearML

Every run also syncs to ClearML (unless started with `--no-clearml`). To get these charts there:

1. Open [app.clear.ml](https://app.clear.ml) → **Projects** → **Tello** → **onboard-rl**.
2. Tick the `w10_front_*` experiments in the list → click **Compare** in the bar at the bottom.
3. **Scalars** tab:
   * `passes / mean_per_episode` = section 3 above (unsmoothed. Use the *smoothing* slider, ~0.6, to match)
   * `train / std`, `train / approx_kl`, `train / clip_fraction`, `train / explained_variance` = section 4
   * In the Compare view the x-axis is the *total* step count (the parent's 1.7M steps included). This notebook subtracts that, so curves are shifted but the same shape.
4. **Hyperparameters** tab: section `ppo` shows exactly which setting each run changed (gamma, ent_coef, target_kl, ...), and its **Show diff only** toggle hides everything that is the same.

If the ClearML numbers and this notebook's numbers disagree, trust the run folder (it is the source both read from) and tell Claude which chart differs.